# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset DOI: [10.71728/senscience.vq4a-28xa](https://sen.science/doi/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure mlcroissant library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset schema.


In [ ]:
# List record sets and their fields using @id
print("Record sets in this dataset:")
for record_set in dataset.record_sets:
    print(f"  @id: {record_set.id_}, name: {record_set.name}")
    print("    Fields:")
    for field in record_set.fields:
        print(f"      @id: {field.id_}, name: {field.name}, type: {field.data_type}")
    print()

# Choose a record set to further explore
selected_record_set_id = dataset.record_sets[0].id_ if dataset.record_sets else None
if selected_record_set_id:
    print(f"Example records from record set {selected_record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=selected_record_set_id)):
        print(record)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id_ for rs in dataset.record_sets]
print("Record set @ids:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    # Retrieve and load all records for each record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set {record_set_id}")
    else:
        print(f"No records found for record set {record_set_id}")

if record_set_ids:
    # Display columns and preview for the first record set
    print(f"\nColumns in {record_set_ids[0]}:")
    print(dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.


In [ ]:
# Example: Select a numeric field for analysis from the first record set
import numpy as np

target_record_set_id = record_set_ids[0] if record_set_ids else None
if target_record_set_id:
    df = dataframes[target_record_set_id]
    # Find a numeric column
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns in {target_record_set_id}: {numeric_cols}")

    if numeric_cols:
        numeric_field = numeric_cols[0]

        # Set a threshold for demonstration
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 0
        print(f"Using threshold: {threshold:.2f}\n")

        # Filter records where the value exceeds this threshold
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the selected numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a non-numeric field if exists
        non_numeric_cols = [c for c in df.columns if c != numeric_field and df[c].dtype == 'object']
        group_field = non_numeric_cols[0] if non_numeric_cols else None

        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric fields found for EDA in this record set.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id and numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in {target_record_set_id}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

    if len(numeric_cols) > 1:
        # Scatter plot of first two numeric fields
        plt.figure(figsize=(6,6))
        sns.scatterplot(x=df[numeric_cols[0]], y=df[numeric_cols[1]])
        plt.title(f"Scatter plot: {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.xlabel(numeric_cols[0])
        plt.ylabel(numeric_cols[1])
        plt.tight_layout()
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the dataset metadata and records using the `mlcroissant` library referencing all entities by their `@id`.
- We explored available record sets and fields by their `@id`s.
- We performed basic EDA: filtering, normalizing, and grouping numeric data.
- Initial visualizations show the distribution of key quantitative features.
- For deeper domain-specific analysis on mass spectrometry results, refer to the field names and scientific documentation in the schema.